In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="kresnik/zeroth_korean", 
                  repo_type="dataset", local_dir="./zeroth_korean")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files: 100%|██████████| 8/8 [00:03<00:00,  2.51it/s]


'/home/ubuntu/zeroth_korean'

In [3]:
files = glob('zeroth_korean/*/*.parquet')
len(files)

7

In [7]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
            })
        
    return data

In [8]:
# data = loop((files[:1], 0))

In [10]:
data = multiprocessing(files, loop, len(files))

100%|██████████| 1/1 [03:18<00:00, 198.17s/it]


In [12]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'zeroth_korean_audio/zeroth_korean-data-train-00003-of-00006_0.mp3',
 'text': '다른 나라보다 비싼 부품값에 공임까지 부풀린다는 불만이 끊임없이 제기된다',
 'speaker': 'zeroth_korean_audio_205'}

In [13]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'zeroth_korean')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 125.34ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  537kB /  537kB, 56.6kB/s  
Processing Files (1 / 1): 100%|██████████|  537kB /  537kB, 55.9kB/s  
New Data Upload: 100%|██████████|  537kB /  537kB, 55.9kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.04s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/d7194df2995ffcc6bf90b9a38c870d7f5c330be9', commit_message='Upload dataset', commit_description='', oid='d7194df2995ffcc6bf90b9a38c870d7f5c330be9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [14]:
audio_files = [d['audio_filename'] for d in data]

with open('zeroth_korean-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [17]:
# !zip -rq zeroth_korean_audio.zip zeroth_korean_audio

In [18]:
# !hf upload malaysia-ai/Multilingual-TTS zeroth_korean_audio.zip --repo-type=dataset

In [4]:
# !zip -rq zeroth_korean_audio_neucodec.zip zeroth_korean_audio_neucodec

In [5]:
# !hf upload malaysia-ai/Multilingual-TTS zeroth_korean_audio_neucodec.zip --repo-type=dataset